In [ ]:
import os
import cv2
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, ResNet152, ResNet101
from tensorflow.keras.applications.resnet import preprocess_input
from skimage import exposure
from skimage.io import imread
from skimage.transform import resize
from skimage.color import rgb2gray
import pandas as pd
from skimage.filters import gabor_kernel
from scipy import ndimage as ndi
from scipy.fftpack import dct
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import os
import tensorflow as tf
import pandas as pd
from PIL import Image
from tensorflow.keras.applications import ResNet152
from tensorflow.keras.layers import GlobalMaxPooling2D

# ============================================================
# Load Pre-trained ResNet-152
# ============================================================
resnet_152 = ResNet152(weights='imagenet', include_top=False)

# ============================================================
# Create Block-wise Feature Model ONLY ONCE
# ============================================================
residual_blocks = [
    resnet_152.get_layer("conv1_relu").output,
    resnet_152.get_layer("conv2_block3_out").output,
    resnet_152.get_layer("conv3_block8_out").output,
    resnet_152.get_layer("conv4_block36_out").output,
    resnet_152.get_layer("conv5_block3_out").output
]

feature_model = tf.keras.Model(
    inputs=resnet_152.input,
    outputs=residual_blocks
)

# ============================================================
# Dataset Directories
# ============================================================
base_dir = r"/kaggle/input/datasets/saurabhsainiji/iarc-trained-balanced-test-untouched/IARC-casewise-spilit-train-balanced-test-untouched/Balancing-226/Fold-wise/Fold-3"

train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Testing")

# ============================================================
# Class Labels
# ============================================================
labels = ['Type - 1', 'Type - 2', 'Type - 3']

# ============================================================
# Check Corrupted Images
# ============================================================
def is_valid_image(img_path):
    try:
        img = Image.open(img_path)
        img.verify()
        return True
    except Exception:
        return False

# ============================================================
# Block-wise Feature Extraction
# ============================================================
def extract_features(image_path, label):

    # Load image
    img = tf.keras.preprocessing.image.load_img(
        image_path,
        target_size=(224, 224)
    )

    # Convert image to array
    x = tf.keras.preprocessing.image.img_to_array(img)

    # Add batch dimension
    x = tf.expand_dims(x, axis=0)

    # Preprocess
    x = tf.keras.applications.resnet50.preprocess_input(x)

    # Extract activations
    activations = feature_model(x)

    # Apply GMP to each block
    pooled_features = []

    for activation in activations:

        gmp_output = GlobalMaxPooling2D()(activation)

        pooled_features.append(gmp_output)

    # Concatenate all block features
    final_features = tf.keras.layers.concatenate(
        pooled_features,
        axis=-1
    )

    # Convert to 1D NumPy array
    final_features = tf.keras.backend.flatten(
        final_features
    ).numpy()

    return final_features, label

# ============================================================
# Directory-Based Feature Extraction
# ============================================================
def extract_features_from_directory(data_dir):

    features_list = []
    labels_list = []
    filenames_list = []

    processed_count = 0
    corrupted_count = 0

    for label_idx, label in enumerate(labels):

        label_path = os.path.join(data_dir, label)

        if not os.path.exists(label_path):
            print(f"⚠️ Warning: directory not found {label_path}")
            continue

        print(f"\n📂 Processing Class: {label}")

        for root, _, files in os.walk(label_path):

            for file in files:

                # Only image files
                if not (
                    file.lower().endswith('.jpg')
                    or file.lower().endswith('.jpeg')
                    or file.lower().endswith('.png')
                ):
                    continue

                img_path = os.path.join(root, file)

                # -------------------------------------------------
                # Skip Corrupted Images
                # -------------------------------------------------
                if not is_valid_image(img_path):
                    corrupted_count += 1
                    continue

                try:

                    features, lab = extract_features(
                        img_path,
                        label_idx
                    )

                    features_list.append(features)
                    labels_list.append(lab)
                    filenames_list.append(file)

                    processed_count += 1

                    if processed_count % 50 == 0:
                        print(
                            f"✅ Processed Images: {processed_count}"
                        )

                except Exception:
                    corrupted_count += 1
                    continue

    print("\n======================================")
    print(f"✅ Total Processed Images : {processed_count}")
    print(f"❌ Total Corrupted Images: {corrupted_count}")
    print("======================================")

    df = pd.DataFrame({
        'image_name': filenames_list,
        'features': features_list,
        'labels': labels_list
    })

    return df

# ============================================================
# TRAIN FEATURE EXTRACTION
# ============================================================
print("\n🚀 Extracting TRAIN Features...\n")

res_152_train_features_df = extract_features_from_directory(
    train_dir
)

# ============================================================
# TEST FEATURE EXTRACTION
# ============================================================
print("\n🚀 Extracting TEST Features...\n")

res_152_test_features_df = extract_features_from_directory(
    test_dir
)

# ============================================================
# DISPLAY RESULTS
# ============================================================
print("\n======================================")
print("Training Features Shape :", res_152_train_features_df.shape)
print("Testing Features Shape  :", res_152_test_features_df.shape)

print(
    "Unique Training Feature Length:",
    res_152_train_features_df['features'].apply(len).unique()
)

print(
    "Unique Testing Feature Length:",
    res_152_test_features_df['features'].apply(len).unique()
)

print("\nSample Training Data:")
print(res_152_train_features_df.head())

# ============================================================
# SAVE FEATURES
# ============================================================
res_152_train_features_df.to_pickle(
    "resnet152_blockwise_gmp_train_features.pkl"
)

res_152_test_features_df.to_pickle(
    "resnet152_blockwise_gmp_test_features.pkl"
)

print("\n✅ Feature files saved successfully!")

In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, precision_score, f1_score, accuracy_score

# --- ✅ Separate features, labels, and image names for training and testing sets ---
X_train = np.stack(res_152_train_features_df['features'].values)
y_train = np.array(res_152_train_features_df['labels'].values)

X_test = np.stack(res_152_test_features_df['features'].values)
y_test = np.array(res_152_test_features_df['labels'].values)
image_names = np.array(res_152_test_features_df['image_name'].values)  # store filenames

print("ResNet50 features with SVM (Train/Test evaluation)")

# --- ✅ Initialize and Train SVM ---
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

# --- ✅ Predict on test data ---
y_pred = svm.predict(X_test)

# --- ✅ Compute confusion matrix ---
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", conf_matrix)

# --- ✅ Compute metrics ---
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
sensitivity = np.mean([
    conf_matrix[i, i] / conf_matrix[i, :].sum() if conf_matrix[i, :].sum() > 0 else 0
    for i in range(conf_matrix.shape[0])
])
specificity = np.mean([
    (np.sum(conf_matrix) - np.sum(conf_matrix[i, :]) - np.sum(conf_matrix[:, i]) + conf_matrix[i, i]) /
    (np.sum(conf_matrix) - np.sum(conf_matrix[:, i])) if np.sum(conf_matrix[:, i]) > 0 else 0
    for i in range(conf_matrix.shape[0])
])
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# --- ✅ Print overall results ---
print(f"\nAccuracy: {accuracy:.6f}")
print(f"Specificity: {specificity:.6f}")
print(f"Sensitivity (Recall): {sensitivity:.6f}")
print(f"Precision: {precision:.6f}")
print(f"F1 Score: {f1:.6f}")

# --- ✅ Identify and print misclassified images ---
print("\nMisclassified Images:")
misclassified_indices = np.where(y_test != y_pred)[0]

if len(misclassified_indices) == 0:
    print("🎉 No misclassifications! All predictions are correct.")
else:
    for idx in misclassified_indices:
        true_label = labels[y_test[idx]]
        pred_label = labels[y_pred[idx]]
        img_name = image_names[idx]
        print(f"Image: {img_name} | True: {true_label} | Predicted: {pred_label}")
